KMS and Secrets Manager are AWS's core security services for encryption and secrets management. KMS — Key Management Service — creates and controls the cryptographic keys used to encrypt data across AWS. Secrets Manager stores, rotates, and retrieves sensitive credentials so applications never hard-code passwords or API keys. Understanding both services is essential for the SAA-C03 exam and for designing secure architectures on AWS.

## AWS KMS Overview

KMS is a managed service for creating and controlling **cryptographic keys**. It integrates with virtually every AWS service that stores or transmits data — S3, EBS, RDS, DynamoDB, Secrets Manager, SNS, SQS, Lambda, and more.

### Why KMS?
- Keys never leave KMS unencrypted — all cryptographic operations happen inside KMS's FIPS 140-2 validated hardware security modules (HSMs)
- Fine-grained access control via **key policies** and IAM
- Full audit trail — every KMS API call is logged to **CloudTrail**
- Centralised key management across all AWS services

### Key types by ownership

| Type | Who creates it | Who controls it | Cost | Rotation |
|---|---|---|---|---|
| **AWS managed key** | AWS (on your behalf) | AWS | Free | Automatic every year |
| **Customer managed key (CMK)** | You | You | $1/month + API calls | Optional (annual or on-demand) |
| **AWS owned key** | AWS (shared across accounts) | AWS | Free | AWS manages |

> AWS managed keys are created automatically when you enable encryption on a service (e.g., `aws/s3`, `aws/ebs`, `aws/rds`). You cannot see or modify their key policy. Customer managed keys give you full control — custom key policies, grants, manual rotation, cross-account sharing, and the ability to disable or delete them.

## Symmetric vs Asymmetric Keys

| | Symmetric (AES-256) | Asymmetric (RSA / ECC) |
|---|---|---|
| Keys | Single key for encrypt + decrypt | Public key (encrypt/verify) + Private key (decrypt/sign) |
| KMS usage | All AWS service integrations use symmetric keys | Digital signatures, public-key encryption |
| Call KMS to decrypt? | Yes — key never leaves KMS | Private key never leaves KMS; public key downloadable |
| Use case | Server-side encryption across AWS | Code signing, JWT verification, external-party encryption |

> **Exam tip:** When a question mentions encrypting data at rest in S3, EBS, RDS, etc. → always symmetric KMS key. Asymmetric appears for digital signatures and scenarios where an external party encrypts data using your public key.

## Envelope Encryption

KMS does not encrypt large data directly — there is a 4 KB limit per `Encrypt` API call. Instead, AWS uses **envelope encryption**:

```text
1. GenerateDataKey(KeyId)  ──▶  KMS returns:
                                  - Plaintext DEK  (Data Encryption Key)
                                  - Encrypted DEK  (encrypted with your KMS key)

2. Encrypt your data locally using the Plaintext DEK

3. Store: Encrypted data + Encrypted DEK together
   Discard: Plaintext DEK

To decrypt:
4. Send Encrypted DEK to KMS → KMS returns Plaintext DEK
5. Decrypt data locally using Plaintext DEK
```

**Why?**
- Large data is encrypted locally (fast, no KMS latency per byte)
- Only the small DEK travels to KMS — the data never leaves your system
- The encrypted DEK is safe to store alongside the data — useless without KMS access

All AWS service integrations (S3 SSE-KMS, EBS encryption, RDS encryption) use envelope encryption under the hood.

## Key Policies and Access Control

**Key policies** are the primary access control mechanism for KMS keys — unlike other AWS resources, **IAM policies alone are not sufficient**. A KMS key's key policy must explicitly allow the IAM principal access.

### Key policy structure
```json
{
  "Statement": [
    {
      "Sid": "Enable IAM policies",
      "Effect": "Allow",
      "Principal": {"AWS": "arn:aws:iam::123456789012:root"},
      "Action": "kms:*",
      "Resource": "*"
    },
    {
      "Sid": "Allow app to use key",
      "Effect": "Allow",
      "Principal": {"AWS": "arn:aws:iam::123456789012:role/AppRole"},
      "Action": ["kms:Decrypt", "kms:GenerateDataKey"],
      "Resource": "*"
    }
  ]
}
```

The first statement enables IAM — without it, no IAM policies for this key take effect.

### KMS Grants
Grants allow you to **delegate specific key permissions** to AWS services or principals without modifying the key policy:
- Used internally by AWS services (e.g., EBS grants KMS permission to use a key for a specific volume)
- Can be created and retired programmatically
- Useful for cross-account temporary access

### Cross-account access
To use a KMS key from another account:
1. Key policy in Account A must allow Account B's principal
2. IAM policy in Account B must allow `kms:Decrypt` on the key ARN

## Key Rotation, Deletion, and Multi-Region Keys

### Key rotation
- **Automatic rotation**: enable on a CMK — KMS rotates the backing key material every year. The key ID, ARN, and aliases remain the same. Data encrypted with old key material is automatically decryptable (KMS keeps all old versions).
- **On-demand rotation**: rotate immediately without waiting for the annual schedule
- AWS managed keys rotate automatically every year; you cannot disable this

### Key deletion
- KMS keys cannot be deleted immediately — there is a mandatory **waiting period of 7–30 days**
- During the waiting period, the key is disabled but not yet deleted; you can cancel deletion
- Once deleted, any data encrypted with that key is **permanently unrecoverable** — verify no data depends on the key before deleting
- To temporarily prevent use without risk: **disable** the key instead of scheduling deletion

### Multi-Region Keys
Multi-Region Keys are a set of KMS keys in different regions that share the same key material and key ID:
```text
Primary key (us-east-1)  ←──▶  Replica key (eu-west-1)
Same key material, same key ID prefix: mrk-...
```
- Encrypt in one region, decrypt in another — no re-encryption needed
- Use case: global applications, multi-region DynamoDB Global Tables, disaster recovery where encrypted data is replicated cross-region
- Each replica is an independent key with its own key policy

## KMS Integration with AWS Services

### S3 encryption options

| Option | Key management | Notes |
|---|---|---|
| **SSE-S3** | AWS manages keys entirely | Default; no KMS involvement |
| **SSE-KMS** | Your KMS key (AWS managed or CMK) | CloudTrail logs every decryption; CMK gives key policy control |
| **SSE-C** | You provide the key per request | AWS encrypts/decrypts but never stores your key |
| **Client-side** | You encrypt before upload | AWS never sees plaintext |

> **SSE-KMS caveat:** Every S3 GET request triggers a KMS `Decrypt` API call. For high-throughput buckets, this can hit KMS API rate limits (default 5,500–30,000 requests/s per region). Use **S3 Bucket Keys** to reduce KMS calls — S3 generates a short-lived data key from KMS and uses it for many objects, dramatically reducing API call volume.

### EBS encryption
- Enable at volume creation; uses a CMK or the default `aws/ebs` key
- Snapshots of encrypted volumes are automatically encrypted
- Copying a snapshot to another region requires re-encrypting with a key in the destination region
- Account-level default encryption: all new EBS volumes encrypted automatically

### RDS encryption
- Enabled at DB instance creation — cannot be added to an existing unencrypted instance
- Encrypts storage, automated backups, read replicas, and snapshots
- To encrypt an existing unencrypted RDS instance: snapshot → copy snapshot with encryption enabled → restore from encrypted snapshot

## AWS Secrets Manager

Secrets Manager stores, rotates, and retrieves sensitive credentials — database passwords, API keys, OAuth tokens — so applications retrieve secrets at runtime instead of embedding them in code or config files.

### How it works
```text
Application ──GetSecretValue──▶ Secrets Manager ──▶ returns secret JSON
                                       │
                              secret encrypted at rest
                              with KMS key
```

### Automatic rotation
Secrets Manager can rotate secrets **automatically** on a schedule:
1. Secrets Manager invokes a **Lambda rotation function**
2. Lambda creates a new credential in the target service (e.g., new RDS password)
3. Lambda updates the secret value in Secrets Manager
4. Lambda tests the new credential
5. Lambda retires the old credential

- Built-in rotation Lambda templates for: RDS (MySQL, PostgreSQL, Oracle, SQL Server, Aurora), Redshift, DocumentDB
- Custom rotation Lambda for any other secret type
- Rotation schedule: days interval or cron expression

### Secret versioning
Secrets Manager keeps multiple versions with staging labels:
- `AWSCURRENT` — the active version applications retrieve by default
- `AWSPENDING` — the new version during rotation
- `AWSPREVIOUS` — the previous version (rollback)

During rotation there is always a valid current version — zero downtime.

### Cross-region replication
Secrets can be replicated to multiple regions — the replica stays in sync with the primary. Use for: multi-region applications, disaster recovery, read-local patterns.

## Secrets Manager vs SSM Parameter Store

| | Secrets Manager | SSM Parameter Store |
|---|---|---|
| **Cost** | $0.40/secret/month + API calls | Free (Standard); $0.05/parameter/month (Advanced) |
| **Secret rotation** | Built-in automatic rotation with Lambda | Manual only (no built-in rotation) |
| **Encryption** | Always encrypted with KMS | Optional KMS encryption (SecureString) |
| **Secret size** | Up to 65 KB | 4 KB (Standard) / 8 KB (Advanced) |
| **Versioning** | Yes — AWSCURRENT / AWSPENDING / AWSPREVIOUS | Yes — version numbers |
| **Cross-region replication** | Yes | No |
| **Use case** | Database credentials, API keys needing auto-rotation | App config, feature flags, non-rotating secrets |
| **RDS integration** | Native — Secrets Manager manages RDS password lifecycle | Manual |

> **Exam rule:** Automatic rotation required → Secrets Manager. Simple config storage, cost-sensitive, no rotation needed → SSM Parameter Store.

## Working with KMS and Secrets Manager via boto3

In [ ]:
import boto3
import base64
import json

kms     = boto3.client('kms',             region_name='us-east-1')
secrets = boto3.client('secretsmanager',  region_name='us-east-1')

In [ ]:
# Create a customer managed KMS key
key = kms.create_key(
    Description='Production app encryption key',
    KeyUsage='ENCRYPT_DECRYPT',      # symmetric key
    KeySpec='SYMMETRIC_DEFAULT',     # AES-256
    Tags=[{'TagKey': 'Environment', 'TagValue': 'production'}]
)
key_id  = key['KeyMetadata']['KeyId']
key_arn = key['KeyMetadata']['Arn']
print(f"Key ID:  {key_id}")
print(f"Key ARN: {key_arn}")

# Create a friendly alias
kms.create_alias(
    AliasName='alias/prod-app-key',
    TargetKeyId=key_id
)

# Enable automatic annual rotation
kms.enable_key_rotation(KeyId=key_id)
print("Automatic key rotation enabled")

In [ ]:
# Envelope encryption — generate a data key
response = kms.generate_data_key(
    KeyId='alias/prod-app-key',
    KeySpec='AES_256'
)

plaintext_dek  = response['Plaintext']       # use this to encrypt data locally
encrypted_dek  = response['CiphertextBlob']  # store this alongside encrypted data

# Simulate encrypting data with the plaintext DEK (using Python's cryptography lib)
print(f"Plaintext DEK length:  {len(plaintext_dek)} bytes")
print(f"Encrypted DEK length:  {len(encrypted_dek)} bytes")
print("Store encrypted_dek with your data; discard plaintext_dek after use")

# To decrypt later: call KMS with the encrypted DEK
decrypt_response = kms.decrypt(CiphertextBlob=encrypted_dek)
recovered_dek = decrypt_response['Plaintext']  # use to decrypt data
print(f"DEK recovered: {plaintext_dek == recovered_dek}")

In [ ]:
# Direct encrypt/decrypt (small values only — up to 4 KB)
plaintext = b'sensitive config value'

encrypted = kms.encrypt(
    KeyId='alias/prod-app-key',
    Plaintext=plaintext
)
ciphertext = encrypted['CiphertextBlob']
print(f"Encrypted: {base64.b64encode(ciphertext).decode()[:40]}...")

decrypted = kms.decrypt(CiphertextBlob=ciphertext)
print(f"Decrypted: {decrypted['Plaintext']}")

In [ ]:
# Create a secret in Secrets Manager
secret = secrets.create_secret(
    Name='prod/app/db-credentials',
    Description='RDS Aurora production credentials',
    KmsKeyId='alias/prod-app-key',   # use our CMK to encrypt the secret
    SecretString=json.dumps({
        'username': 'appuser',
        'password': 'initial-password-123!',
        'host':     'prod-cluster.cluster-abc.us-east-1.rds.amazonaws.com',
        'port':     5432,
        'dbname':   'appdb'
    })
)
print(f"Secret ARN: {secret['ARN']}")

In [ ]:
# Retrieve a secret at application runtime
response = secrets.get_secret_value(SecretId='prod/app/db-credentials')
credentials = json.loads(response['SecretString'])

print(f"Host:     {credentials['host']}")
print(f"Username: {credentials['username']}")
# Use credentials['password'] to connect — never hard-code it

# In production: cache the secret and refresh on rotation
# The AWS SDK Secrets Manager caching client handles this automatically

In [ ]:
# Enable automatic rotation (requires a rotation Lambda to be deployed)
secrets.rotate_secret(
    SecretId='prod/app/db-credentials',
    RotationLambdaARN='arn:aws:lambda:us-east-1:123456789012:function:SecretsManagerRDSPostgreSQLRotationSingleUser',
    RotationRules={
        'AutomaticallyAfterDays': 30,    # rotate every 30 days
    },
    RotateImmediately=True               # also rotate now
)
print("Automatic rotation enabled — secret will rotate every 30 days")

In [ ]:
# SSM Parameter Store — for non-rotating config values
ssm = boto3.client('ssm', region_name='us-east-1')

# Store an encrypted parameter (SecureString)
ssm.put_parameter(
    Name='/prod/app/feature-flag',
    Value='true',
    Type='String',        # String, StringList, or SecureString (KMS encrypted)
    Overwrite=True
)

ssm.put_parameter(
    Name='/prod/app/api-key',
    Value='sk-abc123',
    Type='SecureString',  # encrypted with KMS
    KeyId='alias/prod-app-key',
    Overwrite=True
)

# Retrieve
param = ssm.get_parameter(Name='/prod/app/api-key', WithDecryption=True)
print(f"API key: {param['Parameter']['Value']}")

## Summary

| Concept | Key Takeaway |
|---|---|
| KMS | Managed HSM-backed key service; keys never leave KMS unencrypted |
| AWS managed key | Created by AWS per service (`aws/s3`, `aws/ebs`); free; auto-rotates yearly |
| Customer managed key (CMK) | You create and control; $1/month; custom key policies; can disable/delete |
| Symmetric key | AES-256; single key; used by all AWS service integrations |
| Asymmetric key | RSA/ECC; public + private key pair; digital signatures, external encryption |
| Envelope encryption | GenerateDataKey → encrypt data locally → store encrypted DEK with data |
| Key policy | Primary access control; IAM alone insufficient; must enable IAM in key policy |
| KMS grants | Delegate permissions programmatically without editing key policy |
| Key rotation | Annual automatic rotation; key ID unchanged; old material retained for decryption |
| Key deletion | 7–30 day waiting period; unrecoverable after deletion; disable instead if unsure |
| Multi-region keys | Same key material across regions; encrypt in one region, decrypt in another |
| S3 SSE-KMS | KMS decryption on every GET; use S3 Bucket Keys to reduce KMS API calls |
| S3 Bucket Keys | Short-lived data key per bucket reduces KMS call volume dramatically |
| EBS encryption | Enabled at creation; snapshots inherit encryption; account-level default available |
| RDS encryption | Enabled at creation only; encrypt existing: snapshot → copy with encryption → restore |
| Secrets Manager | Stores credentials; automatic rotation via Lambda; KMS encrypted |
| Secret rotation | Lambda rotates on schedule: create → update → test → retire old credential |
| Secret versioning | AWSCURRENT / AWSPENDING / AWSPREVIOUS; zero-downtime rotation |
| Cross-region replication | Secrets replicated to other regions; use for multi-region apps and DR |
| Secrets Manager vs SSM | Secrets Manager: auto-rotation, RDS native, higher cost; SSM: config, free tier, no rotation |